In [1]:
# 데이터 확인하기 2025.11.19
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

In [2]:
# 원본 데이터 불러오기
df = pd.read_csv("../data/train.csv")
df_test = pd.read_csv("../data/test.csv")

In [3]:
remove_cols = pd.read_csv('../doc/remove_cols.xls', header=0).squeeze()
df.drop(columns=remove_cols, axis=1, inplace=True)
df.drop(columns=['ID', 'TARGET'], axis=1, inplace=True) # train 데이터 ID와 TARGET 모두 제거
df.columns

Index(['var3', 'var15', 'imp_ent_var16_ult1', 'imp_op_var39_comer_ult1',
       'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult1',
       'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1',
       'imp_op_var41_efect_ult3', 'imp_op_var41_ult1',
       ...
       'saldo_medio_var8_ult3', 'saldo_medio_var12_hace2',
       'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1',
       'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2',
       'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1',
       'saldo_medio_var13_corto_ult3', 'var38'],
      dtype='object', length=149)

In [4]:
remove_cols_test = pd.read_csv('../doc/remove_cols.xls', header=0).squeeze()
df_test.drop(columns=remove_cols, axis=1, inplace=True)
df_test.drop(columns=['ID'], axis=1, inplace=True) # test 데이터 ID와 TARGET 모두 제거

In [5]:
print(df.shape) # (76020, 154)
desc = df.describe()
desc.to_csv('../data/df_describe_after_removed_cols.csv')
desc.loc['min']

(76020, 149)


var3                            -999999.00
var15                                 5.00
imp_ent_var16_ult1                    0.00
imp_op_var39_comer_ult1               0.00
imp_op_var39_comer_ult3               0.00
                                   ...    
saldo_medio_var13_corto_hace2         0.00
saldo_medio_var13_corto_hace3         0.00
saldo_medio_var13_corto_ult1          0.00
saldo_medio_var13_corto_ult3          0.00
var38                              5163.75
Name: min, Length: 149, dtype: float64

In [6]:
print(df_test.shape) # (76020, 154)
desc = df_test.describe()
desc.to_csv('../data/df_describe_after_removed_cols.csv')
desc.loc['min']

(75818, 149)


var3                            -999999.00
var15                                 5.00
imp_ent_var16_ult1                    0.00
imp_op_var39_comer_ult1               0.00
imp_op_var39_comer_ult3               0.00
                                   ...    
saldo_medio_var13_corto_hace2         0.00
saldo_medio_var13_corto_hace3         0.00
saldo_medio_var13_corto_ult1          0.00
saldo_medio_var13_corto_ult3          0.00
var38                              1202.73
Name: min, Length: 149, dtype: float64

In [7]:
# 3. IQR 기반 이상치 기준 계산
Q1 = desc.loc["25%"]
Q3 = desc.loc["75%"]
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


In [8]:
# 이상치 마스킹 (컬럼별로)
outlier_mask = pd.DataFrame(False, index=df.index, columns=df.columns)

for col in df.select_dtypes(include="number").columns:
    if col in lower_bound.index:
        lb = lower_bound[col]
        ub = upper_bound[col]
        outlier_mask[col] = (df[col] < lb) | (df[col] > ub)

# 5. 이상치 기준 + 마스크를 outlier_bounds에 통합
outlier_bounds = pd.DataFrame({
    "min" : desc.loc['min'],
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "max": desc.loc['max'],
    "LowerBound": lower_bound,
    "UpperBound": upper_bound,
    "OutlierCount": outlier_mask.sum()
})

print(outlier_bounds)


                                     min          Q1          Q3         IQR  \
var3                          -999999.00      2.0000       2.000      0.0000   
var15                               5.00     23.0000      39.000     16.0000   
imp_ent_var16_ult1                  0.00      0.0000       0.000      0.0000   
imp_op_var39_comer_ult1             0.00      0.0000       0.000      0.0000   
imp_op_var39_comer_ult3             0.00      0.0000       0.000      0.0000   
...                                  ...         ...         ...         ...   
saldo_medio_var13_corto_hace2       0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_hace3       0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_ult1        0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_ult3        0.00      0.0000       0.000      0.0000   
var38                            1202.73  67549.6125  118315.905  50766.2925   

                                       

In [9]:
# 이상치 마스킹 (컬럼별로) -- test
outlier_mask = pd.DataFrame(False, index=df_test.index, columns=df_test.columns)

for col in df_test.select_dtypes(include="number").columns:
    if col in lower_bound.index:
        lb = lower_bound[col]
        ub = upper_bound[col]
        outlier_mask[col] = (df_test[col] < lb) | (df_test[col] > ub)

# 5. 이상치 기준 + 마스크를 outlier_bounds에 통합
outlier_bounds = pd.DataFrame({
    "min" : desc.loc['min'],
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "max": desc.loc['max'],
    "LowerBound": lower_bound,
    "UpperBound": upper_bound,
    "OutlierCount": outlier_mask.sum()
})

print(outlier_bounds)


                                     min          Q1          Q3         IQR  \
var3                          -999999.00      2.0000       2.000      0.0000   
var15                               5.00     23.0000      39.000     16.0000   
imp_ent_var16_ult1                  0.00      0.0000       0.000      0.0000   
imp_op_var39_comer_ult1             0.00      0.0000       0.000      0.0000   
imp_op_var39_comer_ult3             0.00      0.0000       0.000      0.0000   
...                                  ...         ...         ...         ...   
saldo_medio_var13_corto_hace2       0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_hace3       0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_ult1        0.00      0.0000       0.000      0.0000   
saldo_medio_var13_corto_ult3        0.00      0.0000       0.000      0.0000   
var38                            1202.73  67549.6125  118315.905  50766.2925   

                                       

In [10]:
# 데이터 전처리 : 
# var3 feature -> -999999 -> 최빈값인 2로 대체
print(df.var3.value_counts())
df['var3'] = df['var3'].replace(-999999 , 2, inplace=False) # False 변경된 내용 변환

var3
 2         74165
 8           138
-999999      116
 9           110
 3           108
           ...  
 63            1
 194           1
 40            1
 57            1
 87            1
Name: count, Length: 208, dtype: int64


In [11]:
# 데이터 전처리 : 
# 1. var3 feature -> -999999 -> 최빈값인 2로 대체
print(df_test.var3.value_counts())
df_test['var3'] = df_test['var3'].replace(-999999 , 2, inplace=False) # False 변경된 내용 변환

var3
 2         73962
-999999      120
 8           116
 9           108
 3           107
           ...  
 127           1
 51            1
 194           1
 87            1
 225           1
Name: count, Length: 199, dtype: int64


In [12]:
outlier_bounds.to_csv('../data/outlier_bounds_154cols.csv')

In [13]:
from matplotlib import rc

plt.rc('font', family='Malgun Gothic')

In [17]:
from sklearn.preprocessing import StandardScaler

# 숫자형 컬럼만 선택
num_cols = df.select_dtypes(include="number").columns

# 스케일러 객체 생성
scaler = StandardScaler()

# fit + transform
df[num_cols] = scaler.fit_transform(df[num_cols])

# 확인
print(df[num_cols].value_counts())

var3        var15      imp_ent_var16_ult1  imp_op_var39_comer_ult1  imp_op_var39_comer_ult3  imp_op_var41_comer_ult1  imp_op_var41_comer_ult3  imp_op_var41_efect_ult1  imp_op_var41_efect_ult3  imp_op_var41_ult1  imp_op_var39_efect_ult1  imp_op_var39_efect_ult3  imp_op_var39_ult1  ind_var1_0  ind_var5_0  ind_var5   ind_var8_0  ind_var8  ind_var12_0  ind_var12  ind_var13_0  ind_var13_corto_0  ind_var13_corto  ind_var13_largo_0  ind_var13  ind_var14_0  ind_var24_0  ind_var24  ind_var25_cte  ind_var26_0  ind_var26_cte  ind_var26  ind_var25_0  ind_var25  ind_var30_0  ind_var30  ind_var37_cte  ind_var37_0  ind_var37  ind_var39_0  ind_var40_0  ind_var41_0  num_var1_0  num_var4   num_var5_0  num_var5   num_var8_0  num_var8  num_var12_0  num_var12  num_var13_0  num_var13_corto_0  num_var13_corto  num_var13_largo_0  num_var13  num_var14_0  num_var24_0  num_var24  num_var26_0  num_var26  num_var25_0  num_var25  num_op_var41_hace2  num_op_var41_hace3  num_op_var41_ult1  num_op_var41_ult3  num_op_v